# SQL Teil 5: Eigene Erweiterung

Arbeitsnotebook zu [Anforderungen.md](Anforderungen.md). Lest dort zuerst die Phasen und die Checkliste.

So arbeitet ihr:

1. Ihr schreibt eure Änderungen in [migration.sql](migration.sql) und eure Testdaten in [testdaten.sql](testdaten.sql).
2. Die Zelle **Neu aufbauen** nimmt eine frische Kopie der Referenzdatenbank, führt beide Skripte aus und prüft, ob dabei Daten verloren gegangen sind.
3. Danach testet und fragt ihr in diesem Notebook ab.

Nach jeder Änderung an den Skripten: **Neu aufbauen** ausführen, dann die Zellen darunter.

# Setup

Baut beim ersten Mal die Vorlage `vorlage.db` aus der Referenz und definiert die Funktion `neu_aufbauen()`.

In [ ]:
import sqlite3
from pathlib import Path
from sql.connection import ConnectionManager

VORLAGE = Path("vorlage.db")
DB = Path("erweiterung.db")

if not VORLAGE.exists():
    tmp = Path("vorlage.tmp")
    tmp.unlink(missing_ok=True)
    con = sqlite3.connect(tmp)
    con.executescript(Path("../referenz/setup.sql").read_text(encoding="utf-8"))
    con.close()
    tmp.rename(VORLAGE)


def zaehlen(con):
    """Zählt die Zeilen in allen Tabellen der Referenz."""
    anzahl = {}
    for (t,) in VORLAGE_TABELLEN:
        try:
            anzahl[t] = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        except sqlite3.Error:
            anzahl[t] = "Tabelle fehlt"
    return anzahl


con = sqlite3.connect(VORLAGE)
VORLAGE_TABELLEN = con.execute(
    "SELECT name FROM sqlite_schema WHERE type = 'table' AND name NOT LIKE 'sqlite_%'"
).fetchall()
con.close()


def neu_aufbauen():
    """Frische Kopie der Referenz, dann migration.sql und testdaten.sql ausführen."""
    ConnectionManager.close_all()  # Verbindung des Notebooks schließen, sonst ist die Datei gesperrt

    quelle, ziel = sqlite3.connect(VORLAGE), sqlite3.connect(DB)
    quelle.backup(ziel)
    quelle.close()
    ziel.close()

    con = sqlite3.connect(DB)
    vorher = zaehlen(con)
    for datei in ["migration.sql", "testdaten.sql"]:
        # Die Migration läuft mit ausgeschalteter Fremdschlüsselprüfung, damit ein
        # DROP TABLE nicht über ON DELETE CASCADE Daten in anderen Tabellen löscht.
        # Für die Testdaten ist die Prüfung eingeschaltet.
        con.execute("PRAGMA foreign_keys = " + ("OFF" if datei == "migration.sql" else "ON"))
        try:
            con.executescript(Path(datei).read_text(encoding="utf-8"))
        except sqlite3.Error as fehler:
            con.close()
            print(f"❌ Fehler in {datei}: {fehler}")
            print("   Die Transaktion wurde nicht abgeschlossen, an der Datenbank hat sich durch diese Datei nichts geändert.")
            return
        print(f"✓ {datei} ausgeführt")
        if datei == "migration.sql":
            nachher = zaehlen(con)
            verloren = {t: (vorher[t], nachher[t]) for t in vorher if nachher[t] != vorher[t]}
            if verloren:
                print("⚠️ Die Migration hat bestehende Daten verändert (vorher, nachher):", verloren)
            else:
                print("✓ Keine bestehenden Daten verloren")
            kaputt = con.execute("PRAGMA foreign_key_check").fetchall()
            if kaputt:
                print(f"⚠️ {len(kaputt)} Verweise zeigen ins Leere, z. B. Tabelle {kaputt[0][0]}, Zeile {kaputt[0][1]} → {kaputt[0][2]}")
            else:
                print("✓ Alle Fremdschlüssel-Verweise stimmen")
    con.close()

# Neu aufbauen

In [ ]:
neu_aufbauen()

%load_ext sql
%config SqlMagic.displaylimit = 20
%sql sqlite:///erweiterung.db
%sql PRAGMA foreign_keys = ON;

### Kontrolle: Welche Tabellen, Views und Trigger sind neu?

In [ ]:
%%sql
SELECT type, name
FROM sqlite_schema
WHERE name NOT IN ('account', 'class', 'faction', 'player', 'guild', 'player_guild', 'guild_role',
                   'player_guild_role', 'stat', 'player_stat', 'level', 'player_level', 'sqlite_sequence')
  AND name NOT LIKE 'sqlite_autoindex_%'
ORDER BY type, name;

# Phase 5: Prüfen

## Regeltests

Jede dieser Zellen **muss eine rote Fehlermeldung** liefern. Schreibt darüber, welche Regel ihr testet, und darunter, welche Meldung kam. Führt die Zellen einzeln aus, weil Run All beim ersten Fehler stoppt.

### Regeltest 1: ...

In [ ]:
%%sql
SELECT 'TODO: INSERT oder UPDATE, das scheitern muss' AS aufgabe;

### Regeltest 2: ...

In [ ]:
%%sql
SELECT 'TODO: INSERT oder UPDATE, das scheitern muss' AS aufgabe;

### Regeltest 3: ...

In [ ]:
%%sql
SELECT 'TODO: INSERT oder UPDATE, das scheitern muss' AS aufgabe;

## Fachfragen

Die Fragen aus [meine-erweiterung.md](meine-erweiterung.md). Schreibt die Frage jeweils in die Überschrift.

### Fachfrage 1: ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Fachfrage 2: ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Fachfrage 3: ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Fachfrage 4: ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Fachfrage 5: ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

# Phase 6: Betrieb

## Werkzeuge prüfen

Zeigt, dass eure Werkzeuge wirken: bei einem Index mit `EXPLAIN QUERY PLAN`, bei einem Trigger mit einer Änderung und einem Blick in die betroffene Tabelle, bei einer View mit einer Abfrage darauf.

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

## Datenschutz: Account löschen

Sucht einen Account, der Daten in euren neuen Tabellen hat, und löscht ihn. Prüft danach: Was ist aus euren Tabellen verschwunden, was ist geblieben? Passt das zu dem, was ihr in [meine-erweiterung.md](meine-erweiterung.md) festgelegt habt?

Am Ende einfach **Neu aufbauen** ausführen, dann ist der Account wieder da.

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

# Fertig?

Geht die [Checkliste](Anforderungen.md#checkliste-auch-für-das-review) durch. Dann tauscht ihr mit einem anderen Team für das Review.